In [1]:
cd(joinpath(pwd(), "src/LorentzianSimplexSolver"))

using Pkg
Pkg.activate(".")
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/src/LorentzianSimplexSolver`


In [2]:
include("../../scripts/run_geometry.jl");
include("../../scripts/run_action.jl");
include("../../scripts/run_dlogEh_dX.jl")
include("../../scripts/run_deta_dl.jl")
include("../perturbations/TransverseBasis.jl")
include("../perturbations/Soln_dY_dX.jl")
include("../perturbations/DθDl.jl");

In [3]:
using JLD2

using .RunGeometry
using .RunAction
using .RunDlogEhDX
using .DηDLUtils
using .TransverseBasis
using .Soln_dY_dX

#### Geometry setup

In [4]:
simplices = [[2,3,4,6,7],[1,3,4,6,7],[1,2,4,6,7],[1,2,3,6,7],[1,2,3,4,7],[2,3,5,6,8],[1,3,5,6,8],[1,2,5,6,8],[1,2,3,6,8],[1,2,3,5,8],[2,4,5,6,9],[1,4,5,6,9],[1,2,5,6,9],[1,2,4,6,9],[1,2,4,5,9],[3,4,5,6,10],[1,4,5,6,10],[1,3,5,6,10],[1,3,4,6,10],[1,3,4,5,10],[3,4,5,6,11],[2,4,5,6,11],[2,3,5,6,11],[2,3,4,6,11],[2,3,4,5,11]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

coords_lines = [
"0, 0, 0, 0",
"0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
"0, 0, 0, -3.398088489694245",
"-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
"0, 0, -2.942830956382712, -1.6990442448471226",
"-0.068, -0.27, -0.5, -1.3",
"-0.06165622828269508, -0.7476319083813029, -0.49237746085102824, -1.6192353958776982",
"-0.013600000000000001, -0.6089055267050423, -0.8847549217020566, -1.6192353958776982",
"-0.05471634704156723, -0.6634799624836852, -1.2905133626305172, -1.3266576479993824",
"-0.05996263164968173, -0.18743250119993968, -0.8604521717798855, -1.5747576871100133",
"-0.0423034122998675, -0.5046526423930288, -1.0289336294868097, -2.25080216236223"
]

const ScalarT = Float64
# tol = 1e-10;
# const ScalarT = BigFloat
const tol = parse(ScalarT, "1e-8")

if ScalarT === BigFloat
    setprecision(BigFloat, 80)
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(80)
    # LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
end

gamma_vals = ScalarT(0.1);

vertex_coords = Dict{Int, Vector{ScalarT}}()  

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [5]:
geom = run_geometry_pipeline(simplices, coords_lines, ScalarT, tol);

#### Action and variables calculation

In [6]:
deficit_angles, dihedral_angles, _, _, iRegge = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, simplices, vertex_coords);

In [7]:
γ = LorentzianSimplexSolver.DefineAction.γsym()
sd, S_symbols, phase_soln = RunAction.run_action(geom, dihedral_angles, γ);

g_vars = geom.varias[:g_var]
z_vars = geom.varias[:z_var]
η_vars = geom.varias[:η_var]

vars = vcat(g_vars, z_vars, η_vars);

In [8]:
using SymEngine
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=gamma_vals);

S = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S_symbols, phase_soln);
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
SF_action = SymEngine.expand(S_val)
@show SF_action, iRegge;

(SF_action, iRegge) = (7.41033540377297e-14 - 0.962834958363162*im, 0.0 - 0.0962834958371431im)


#### Computation of $\frac{\partial \log E_h}{\partial X^\alpha}$

In [9]:
nh = length(geom.connectivity[1]["OrderBulkFaces"])

dlogEh_dX_sym = RunDlogEhDX.run_dlogEh_dX(geom)

dlogEh_dX_vals = RunDlogEhDX.evaluate_dlogEh_dX(dlogEh_dX_sym, geom, sd; γval=gamma_vals);

#### Computation of $\frac{\partial \log E_b}{\partial X^\alpha}$, $\frac{\partial \log E_b}{\partial Y^\alpha}$

In [10]:
nb = length(geom.connectivity[1]["OrderBDryFaces"])

dlogEb_dX_sym, dlogEb_dY_sym, Y_vars = RunDlogEhDX.run_dlogEb_dXY(geom);

dlogEb_dX_vals, dlogEb_dY_vals = RunDlogEhDX.evaluate_dlogEb_dXY(dlogEb_dX_sym, dlogEb_dY_sym, Y_vars, geom, sd, phase_soln; γval=gamma_vals);

#### Computation of $\frac{\partial \log k_b E_b}{\partial X^\alpha}$, $\frac{\partial \log k_b E_b}{\partial Y^\alpha}$, $\frac{\partial^2 \log k_b E_b}{\partial X^\alpha \partial Y^\alpha}$ and $\frac{\partial^2 \log k_b E_b}{\partial Y^\alpha}$

In [11]:
dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym = RunDlogEhDX.run_kblogEb_dXY(geom, Y_vars);

In [12]:
dkbEb_dX_vals, dkbEb_dY_vals, d2kbEb_dXdY_vals, d2kbEb_dYdY_vals = RunDlogEhDX.evaluate_kblogEb_all(dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym, geom, sd, Y_vars, phase_soln; γval=gamma_vals);

In [13]:
d2kbEb_dXdY_vals_sumb = sum(d2kbEb_dXdY_vals[i, :, :] for i in 1:nb);

#### Computation of $\frac{\partial \eta_h}{\partial \ell_s}$

In [14]:
# dηdl_matrix is a nh x nl matrix
η_h_vertices = DηDLUtils.get_bulk_faces_vertices(geom)

bulk_edges, bdry_edges = DηDLUtils.get_bulk_edges(geom, η_h_vertices)
bdry_edges_perturb = [bdry_edges[1]] # here we only perturb one boundary edge
perturb_edges = vcat(bulk_edges, bdry_edges_perturb)
    
dηdl_matrix = DηDLUtils.build_dηdl_matrix(η_h_vertices, perturb_edges, vertex_coords, ScalarT, gamma_vals);
nl = length(perturb_edges)
nt = nh - nl;

#### Computation of $\hat{e}^i_h$

In [15]:
# eListHT is a nh x nt matrix
eListHT = TransverseBasis.compute_transverse_basis(dηdl_matrix, tol);

#### Computation of $\frac{\partial k_b}{\partial \ell_s}$ and $\frac{\partial^2 k_b}{\partial \ell_s^2}$

In [16]:
dkbdl, d2kb_dldl = Soln_dY_dX.dkb_dl(geom, nl, bdry_edges_perturb, vertex_coords ;γ=gamma_vals);

#### computation of $\delta\epsilon$ and $\delta\Theta$

In [17]:
DϵDl = DθDl_module.compute_dθDl(simplices, η_h_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

In [18]:
bd_faces = geom.connectivity[1]["OrderBDryFaces"];
kb_vertices = [geom.connectivity[1]["TetFaces"][f[1][1]][f[1][2]][f[1][3]] for f in bd_faces];
DΘDl = DθDl_module.compute_dθDl(simplices, kb_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

#### Hessian matrix computation or read Hessian from files

In [ ]:
H_symbols = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S, vars);
H_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_symbols, sd; γ=gamma_vals);

#### Computation of $B_\alpha$ matrix, $C_\alpha$, $\frac{\partial Y}{\partial \ell}$ matrix and $\frac{\partial X}{\partial \ell}$ matrix

In [ ]:
ng = length(g_vars)
nz = length(z_vars)
nX = ng + nz
X_vars = vcat(g_vars, z_vars)

invHessianXX = inv(H_eval[1:nX, 1:nX]);

In [ ]:
Bα = transpose(transpose(dηdl_matrix) * dlogEh_dX_vals);

In [ ]:
M_matrix = vcat(dlogEb_dY_vals[:, nb+1:end] - dlogEb_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end], -dlogEh_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end]);
dmatrix = vcat(im * gamma_vals/2 * DΘDl + dlogEb_dX_vals * invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl), im * gamma_vals/2 * DϵDl + dlogEh_dX_vals * invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl));
using LinearAlgebra
DξDl = pinv(M_matrix, atol=tol, rtol=tol) * dmatrix;

In [ ]:
DYDl = vcat(dkbdl, DξDl);

In [ ]:
DXDl = -invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl + d2kbEb_dXdY_vals_sumb[:, nb+1:end] * DξDl);

In [ ]:
Cα =d2kbEb_dXdY_vals_sumb * DYDl;

#### Computation of $\delta^2 Y$ matrix

In [ ]:
kb_vals = [ScalarT(subs(Y_vars[i], vals)) for i in 1:nb];

Mm = [kb_vals[b] * dlogEb_dY_vals[b, j] for b in 1:nb, j in nb+1:length(Y_vars)];
R = [dkbdl[i, j] *  transpose(dlogEb_dY_vals[i, nb+1:end]) * DξDl[:, j] + transpose(DXDl[:, j]) * d2kbEb_dXdY_vals[i, :, nb+1:end] * DξDl[:, j] + transpose(DξDl[:, j]) * d2kbEb_dYdY_vals[i, nb+1:end, nb+1:end] * DξDl[:, j] for i in 1:nb, j in 1:nl]

D2ξDl2 = - pinv(Mm, atol=tol, rtol=tol) * R[:, end];

In [ ]:
nxi = length(Y_vars) - nb
D2ξDl2_matrix = [j==k==nl ? D2ξDl2[b] : 0 for b in 1:nxi, j in 1:nl, k in 1:nl]
D2YDl2_matrix = vcat(d2kb_dldl, D2ξDl2_matrix);

#### Computation of boundary linear and quadratic term $\mathcal{I}_{bdry}^{(1)}$ and $\mathcal{I}_{bdry}^{(2)}$

In [ ]:
BA = vcat(Bα + Cα, zeros(nt, nl));

In [ ]:
eta_h = [LorentzianSimplexSolver.DefineSymbols.make_symbol("η_$(faces[1][1])$(faces[1][2])$(faces[1][3])") for faces in geom.connectivity[1]["OrderBulkFaces"]]
eta_h_vals = [ScalarT(subs(eta_h[i], vals)) for i in 1:nh];
Ahh = Matrix(Diagonal(eta_h_vals))

hαβ = H_eval[1:nX, 1:nX] + transpose(dlogEh_dX_vals) * Ahh * dlogEh_dX_vals;
Hαi = transpose(dlogEh_dX_vals) *  eListHT;

HIJ = [hαβ Hαi; transpose(Hαi) zeros(nt, nt)];
invHIJ = inv(HIJ);

In [ ]:
DtXDl = -invHIJ * BA;

In [ ]:
d2kbEb_dYdY_vals_sumb = sum(d2kbEb_dYdY_vals[i, :, :] for i in 1:nb);
dkbEb_dY_vals_sumb = sum(dkbEb_dY_vals[i, :] for i in 1:nb);
Iboundary_linear = transpose(dkbEb_dY_vals_sumb) * DYDl

1×2 transpose(::Vector{Any}) with eltype Any:
 2.37515810267872e-11 - 1.85890881472718e-12*im  …  -5.0159827192416268e-12 - 2.4781224413631529*im

In [ ]:
Iboundary_qadratic = 1/2 * transpose(DYDl) * d2kbEb_dYdY_vals_sumb * DYDl + 1/2 * sum(dkbEb_dY_vals_sumb[i] * D2YDl2_matrix[i, :, :] for i in 1:length(Y_vars))

2×2 Matrix{Basic}:
     -9.37086567783473e-23 + 8.36554899297397e-23*im  …  -5.9989938661579138e-14 - 1.5270595244103224e-12*im
 -5.9989938661578895e-14 - 1.5270595244103227e-12*im           -0.80564880630978431 - 0.82697091702560032*im

#### Computation of Spinfoam quadratic term $(S_{eff} - S^{(0)}_{eff})^{(2)}$

In [ ]:
SF_quadratic = transpose(BA) * DtXDl + 1/2 * transpose(DtXDl) * HIJ * DtXDl + Iboundary_qadratic

2×2 Matrix{Basic}:
  -11.8318208337475 + 0.0699870472705288*im  …   6.6647539844338126 + 6.8507580295286921*im
 6.6647539844338105 + 6.8507580295285832*im     -26.778290268433405 + 6.2620814504642680*im

In [ ]:
SF_quadratic_form2 = -1/2 * transpose(BA[1:nX, :]) * invHIJ[1:nX, 1:nX] * BA[1:nX, :] + Iboundary_qadratic

2×2 Matrix{Basic}:
  -11.8318208337475 + 0.0699870472705601*im  …   6.6647539844338159 + 6.8507580295284615*im
 6.6647539844338035 + 6.8507580295284573*im     -26.778290268431942 + 6.2620814504648380*im

In [ ]:
sum( -1/2 * transpose(BA[1:nX, :]) * invHIJ[1:nX, 1:nX] * BA[1:nX, :] )

-24.474954327001952 + 20.860555473820971*im

In [ ]:
SF_linear_bdry = Iboundary_linear

1×2 transpose(::Vector{Any}) with eltype Any:
 2.37515810267872e-11 - 1.85890881472718e-12*im  …  -5.0159827192416268e-12 - 2.4781224413631529*im

#### Computation of Spinfoam quadratic term from deficit angles $\delta\epsilon$

In [ ]:
ω = - dlogEh_dX_vals * invHessianXX * transpose(dlogEh_dX_vals)
ρ = inv(inv(Ahh) - ω);

In [ ]:
Sij = -transpose(eListHT) * dlogEh_dX_vals * inv(hαβ) * transpose(dlogEh_dX_vals) * eListHT
κ = eListHT * inv(Sij) * transpose(eListHT)

M_kernel =  ρ - ρ * inv(Ahh) * κ * inv(Ahh) * ρ;

In [ ]:
correction_term = - gamma_vals^2/8 * transpose(DϵDl) * M_kernel * DϵDl

2×2 Matrix{ComplexF64}:
 -11.8318+134.159im   6.66475-145.688im
  6.66475-145.688im  -26.7783+124.015im

In [ ]:
iSRegge_quadratic = im*(gamma_vals/4 * transpose(dηdl_matrix) * DϵDl + gamma_vals/4 * (transpose(dkbdl) * DΘDl + sum(dihedral_angles[i] * d2kb_dldl[i, :, :] for i in 1:nb)))

2×2 Matrix{Basic}:
                   -0.0 - 134.08908832298*im  …                     0.0 + 152.539007806566*im
 0.00000000000000000 + 152.53900780656630*im     -0.00000000000000000 - 117.75286743877361*im

In [ ]:
SF_quadratic_wrt_deficit = iSRegge_quadratic + correction_term

2×2 Matrix{Basic}:
   -11.831820833739 + 0.0699870472666362*im  …       6.66475398442444 + 6.85075802952397*im
 6.6647539844244790 + 6.8507580295245905*im     -26.778290268428755 + 6.2620814504656916*im

In [ ]:
iSRegge_linear = im/2 * gamma_vals * transpose(dkbdl) * dihedral_angles

2-element Vector{Any}:
                                           0
 0.00000000000000000 - 2.4781224413627930*im

In [ ]:
sum(iSRegge_linear - transpose(SF_linear_bdry))

-1.8735598307545587e-11 + 2.2187980165002804e-12*im

In [ ]:
sum(SF_quadratic_wrt_deficit - SF_quadratic_form2)

-6.9365208021920921e-12 - 1.1425221879690639e-11*im